In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

from config import (
    TAGS_EXCEL_PATH, DATA_CSV_PART1, DATA_CSV_PART2, TARGET_COL,
    load_tag_lists,
)

# Агрегация: alcohol

Простая сумма TAG + MinMax [0, 1].

In [ ]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)
alcohol_features_list = tag_lists['alcohol_features_list']

part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')
casco_hashtags_full = pd.merge(part1, part2, on='POLICY_ZV', how='inner')
casco_hashtags_full[TARGET_COL] = casco_hashtags_full['CLAIMS_PART_DAM_COUNT'].astype(bool).astype(int)
casco_hashtags_full.set_index('POLICY_ZV', inplace=True)

casco_hashtags_full.drop(columns=['TAG_JOIN_IND'], inplace=True)
casco_hashtags_full['SUM'] = casco_hashtags_full.filter(like='TAG_').fillna(0).sum(axis=1)
casco_hashtags_full = casco_hashtags_full[casco_hashtags_full['SUM'] > 0]
print('shape:', casco_hashtags_full.shape)

## Расчёт alcohol_agg_coef [0, 1]

In [ ]:
ARTIFACTS_DIR = Path('artifacts')
CSV_DIR = ARTIFACTS_DIR / 'csv'
for p in (CSV_DIR,):
    p.mkdir(parents=True, exist_ok=True)

X = casco_hashtags_full.reindex(columns=alcohol_features_list, fill_value=0).fillna(0).astype(float)
score = X.sum(axis=1)
score_norm = MinMaxScaler().fit_transform(score.values.reshape(-1, 1)).flatten()

result = pd.DataFrame({'alcohol_agg_coef': score_norm}, index=casco_hashtags_full.index)
result.to_csv(CSV_DIR / 'alcohol_agg_coef.csv', encoding='utf-8-sig')
print(f'Сохранено: {CSV_DIR / "alcohol_agg_coef.csv"}')
print(result.describe())

## Анализ связи alcohol с claim rate

In [ ]:
# Разбиваем на 10 децилей для визуализации
bins = pd.qcut(result['alcohol_agg_coef'].rank(method='first'), q=10, labels=False)
analysis = (
    pd.DataFrame({'bin': bins, 'target': casco_hashtags_full[TARGET_COL]})
    .groupby('bin')['target']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'claim_rate', 'count': 'objects'})
)
print(analysis)

plt.figure(figsize=(10, 5))
plt.bar(analysis.index.astype(str), analysis['claim_rate'], color='steelblue')
plt.title('alcohol_agg_coef: claim_rate по децилям')
plt.ylabel('claim_rate')
plt.xlabel('дециль')
plt.tight_layout()
plt.show()